**Human Movement Intelligence | Multimodal Movement Learning | Presented by: Reza Saadatyar (2026)**

**01 | NinaPro DB2 Source Pairing and Corrected-Label Audit**

![NinaPro DB2 Source Pairing and Corrected-Label Audit workflow](../../images/01_ninapro_pairing_and_label_audit.svg)

**Research objective:** Determine whether the first non-force exercise provides source-aligned sEMG, accelerometry and corrected movement annotations for a participant.

**Dataset:** [NinaPro Database 2](https://ninapro.hevs.ch/instructions/DB2.html), original subject archives and Atzori et al. (2014).

**Notebook scope:** Inspect real MATLAB variables, label transitions, signal integrity and measurement provenance. No predictive modelling is performed.

**00. Colab Repository Setup**

In [ ]:
# ======================== 00. Colab Repository Setup =========================

# -------------------------- 1. Repository Checkout ---------------------------
from pathlib import Path
import subprocess

repo_url = "https://github.com/RezaSaadatyar/human-movement-intelligence.git"
repo_path = Path("/content") / repo_url.rsplit("/", 1)[-1].removesuffix(".git")
if not repo_path.exists():
    subprocess.run(["git", "clone", "-q", repo_url, str(repo_path)], check=True)

# ---------------------------- 2. Open Repository -----------------------------
%cd {repo_path}


**01. Imports and Research Environment**

In [ ]:
# =================== 01. Imports and Research Environment ====================

# ------------------------ 1. Study Environment Setup -------------------------
from python_modules.core.colab_setup import setup_colab
root = setup_colab("multimodal_movement_learning")

# ---------------------------- 2. Standard Library ----------------------------
from pathlib import Path
import time

# ----------------- 3. Scientific Computing and Data Analysis -----------------
import numpy as np
import pandas as pd

# ------------------- 4. Visualisation and Notebook Display -------------------
import matplotlib.pyplot as plt
from IPython.display import display
from core.plot_style import plot_style, save_figure
from core.output_download import download_outputs_from_colab

plot_style(
    font_family="DejaVu Sans", title_size=12.5, title_weight="normal",
    label_size=11.5, label_weight="normal", tick_size=11,
    legend_size=10, figure_dpi=100, figure_size=(9, 4),
)

# ------------------------ 6. Study Research Utilities ------------------------
from ninapro_source import (
    SOURCE_PAGE, RELEASED_RATE_HZ, archive_path, ensure_subject_archive,
    list_mat_members, load_exercise, inspect_record, contiguous_segments,
)

# ---------------------- 7. Source and Output Locations -----------------------
study_dir = root / "multimodal_movement_learning"
output_dir = study_dir / "output"
processed_dir = root / "data" / "processed" / "ninapro_db2"
raw_dir = root / "data" / "raw" / "ninapro_db2"
output_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
print("Research output directory:", output_dir)


**02. Acquire One Original Participant Archive**

The official DB2 release distributes participant-specific ZIP archives. The acquisition helper reuses a local file or streams the official archive to a temporary download before verifying ZIP integrity. The exercise reader will inspect the first basic-finger movement exercise (E1). No arbitrary ZIP paths are extracted.

In [ ]:
# =============== 02. Acquire One Original Participant Archive ================

NOTEBOOK_NUMBER = 1
PILOT_SUBJECT = 1
EXERCISE_ID = 1
DOWNLOAD_MISSING = True

source = ensure_subject_archive(
    raw_dir, PILOT_SUBJECT, download=DOWNLOAD_MISSING,
)
print("Subject source:", source.name)
print("Archive size (MiB):", round(source.stat().st_size / (1024**2), 2))
print("MATLAB entries:", list_mat_members(source))


**03. Read Paired MATLAB Measurements**

The exercise reader requires 12 sEMG columns, 36 accelerometer columns, equal released sample counts and aligned corrected annotations. The reader does not treat 36 acceleration columns as independent sensors; they represent three axes on each of 12 Trigno modules.

In [ ]:
# ==================== 03. Read Paired MATLAB Measurements ====================

record = load_exercise(source, PILOT_SUBJECT, exercise=EXERCISE_ID)
audit = pd.DataFrame([inspect_record(record)])
display(audit.T.rename(columns={0: "observed value"}))
print("EMG shape:", record.emg.shape, "| Acc shape:", record.acc.shape)
print("Released source-sample rate (Hz):", RELEASED_RATE_HZ)


**04. Inspect Time Base and Source-Level Alignment**

The published release synchronised both modalities onto a shared time grid. A matching row count supports source-grid pairing, but does not imply that the original accelerometer acquired independent samples at 2,000 Hz. Report source-grid seconds without inventing missing native timestamps.

In [ ]:
# ============= 04. Inspect Time Base and Source-Level Alignment ==============

rows = len(record.emg)
alignment_summary = pd.DataFrame([{
    "subject_id": record.subject_id,
    "emg_rows": rows,
    "acc_rows": len(record.acc),
    "corrected_label_rows": len(record.restimulus),
    "repetition_rows": len(record.rerepetition),
    "nominal_duration_s": rows / RELEASED_RATE_HZ,
    "native_acc_sampling_hz_published": 148,
}])
assert alignment_summary[["emg_rows", "acc_rows", "corrected_label_rows", "repetition_rows"]].nunique(axis=1).eq(1).all()
display(alignment_summary)


**05. Audit Corrected Labels and Repetitions**

`restimulus` and `rerepetition` define corrected movement annotation segments. They are reserved for target construction and trial identification, not predictive inputs. Rest is label zero when the source reports it; the verified exercise label set is derived from the MATLAB annotations rather than selected using model performance.

In [ ]:
# ================ 05. Audit Corrected Labels and Repetitions =================

labels, counts = np.unique(record.restimulus, return_counts=True)
label_dictionary = pd.DataFrame({
    "exercise_id": EXERCISE_ID,
    "label": labels.astype(int),
    "source_samples": counts.astype(int),
    "duration_s_nominal": counts / RELEASED_RATE_HZ,
})
segments = contiguous_segments(record.restimulus, record.rerepetition)
segment_frame = pd.DataFrame(segments, columns=[
    "start_sample", "end_sample", "label", "repetition_id",
])
segment_frame["duration_s"] = (segment_frame.end_sample - segment_frame.start_sample) / RELEASED_RATE_HZ
display(label_dictionary)
print("Corrected repetitions:", sorted(np.unique(record.rerepetition).tolist()))
print("Source label segments:", len(segment_frame))
display(segment_frame.groupby("label", as_index=False).agg(segments=("label", "size"),
    median_duration_s=("duration_s", "median")))


**06. Inspect Real Signal Quality and Temporal Context**

Display a short source-indexed segment of one sEMG and one acceleration channel. Both traces are scaled separately for readability and should not be interpreted as amplitude-equivalent physiological quantities. The observed finite-value fraction supports subsequent window eligibility decisions.

In [ ]:
# =========== 06. Inspect Real Signal Quality and Temporal Context ============

quality = pd.DataFrame({
    "modality": ["sEMG", "Acceleration"],
    "channels": [record.emg.shape[1], record.acc.shape[1]],
    "finite_fraction": [np.isfinite(record.emg).mean(), np.isfinite(record.acc).mean()],
})
display(quality)
length = min(len(record.emg), int(RELEASED_RATE_HZ * 2))
t_s = np.arange(length) / RELEASED_RATE_HZ
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, layout="constrained")
axes[0].plot(t_s, record.emg[:length, 0], lw=0.8)
axes[0].set(ylabel="sEMG channel 1 (source units)", title="Paired E1 source interval")
axes[1].plot(t_s, record.acc[:length, 0], lw=0.9)
axes[1].set(xlabel="Time on released common grid (s)", ylabel="Acc column 1 (source units)")
for ax in axes: ax.grid(alpha=0.2)
save_figure(fig, output_dir / "01_alignment_examples.png")
plt.show()


**07. Export Source, Label and Sensor Dictionaries**

The pairing registry, corrected label dictionary and channel inventory are stable upstream inputs for aligned-window construction. Original source signals remain in their acquired files; only source observations and figure files are exported here.

In [ ]:
# ============= 07. Export Source, Label and Sensor Dictionaries ==============

sensors = ([{"modality": "sEMG", "column": f"emg_{i+1:02d}",
             "sensor_module": i+1, "axis": "sEMG"} for i in range(12)]
           + [{"modality": "Acceleration", "column": f"acc_{i*3+j+1:02d}",
               "sensor_module": i+1, "axis": "xyz"[j]} for i in range(12) for j in range(3)])
sensor_dictionary = pd.DataFrame(sensors)
audit.to_csv(output_dir / "01_pairing_registry.csv", index=False)
label_dictionary.to_csv(output_dir / "01_label_dictionary.csv", index=False)
sensor_dictionary.to_csv(output_dir / "01_sensor_dictionary.csv", index=False)
display(sensor_dictionary.groupby("modality", as_index=False).agg(columns=("column", "count")))


**08. Current Outputs and Scientific Interpretation**

The observed exercise fields, labels, sensor columns and pairing registry define the source contract consumed by Notebook 02. The next analysis constructs fixed-duration, trial-safe windows from original source rows.

In [ ]:
# ============= 08. Current Outputs and Scientific Interpretation =============

display(pd.DataFrame({"output_file": [
    p.name for p in sorted(output_dir.glob("%02d_*" % NOTEBOOK_NUMBER))
    if p.is_file()
]}))
download_outputs_from_colab(
    folder="multimodal_movement_learning/output",
    stage=f"{NOTEBOOK_NUMBER:02d}",
)
